# 11 · Projection-free inference: redesign and validation on new synthetic worlds

**Why** (review 4). On NEON summer data the train-only visual projection E[V | hour, T, doy] (D-035) made the contrast unstable (notebook 10 §3). The redesign removes all visual-derived columns from the context. That projection is kept only to reproduce notebook 10.

**The redesigned contrast.**
- **Context model q_C(D | C).** Independently specified, from nested fixed bases (`worlds.add_context_levels`). For each split, the seasonal bases are evaluated at day of year clamped to that split's training days (`worlds.split_levels`), so season is held at its last fitted value and never extrapolated:
  - L1: hour Fourier basis, T spline, T × hour, day-of-year spline (the notebook 10 context);
  - L2: adds a third hour harmonic, T × season products and a finer day-of-year spline;
  - L3: adds the T-spline × hour tensor product.

  For every split, the level is chosen by week-grouped inner cross-validation of the context-only log score on that split's training rows (`inference.select_capacity`). The ridge penalty is chosen in the same way. Visual columns and held-out labels are never read.
- **Joint model q_VC(D | V, C).** The chosen context design plus V, with V clipped to its training rows' 0.5–99.5 % range (`inference.clip_to_training`). Ridge mean, homoscedastic robust Gaussian score (1 % Cauchy), common support on (hour, T).
- **Zero-information test Q0.** The cyclic-shift refitting test with shifts of at least 7 days, within years. Its null generator projects V linearly on the label-free L3 context basis; there is no boosted visual projection.
- **Operational gain G.** A chronological hold-out, with an indicative week-cluster interval.

**Two study designs.**

| design | record | cross-fit folds | chronological hold-out |
|---|---|---|---|
| `1 season` | 92 days (as NEON 2023) | 5 contiguous 16-day blocks, 3-day purge | first 60 % of days, 2-day purge, rest evaluated |
| `3 seasons` | 3 × 92 days (planned NEON extension) | leave one season out | seasons 1–2 fit, season 3 evaluated |

**Targets.**
- The *frozen target* of G is the score of the same fitted models on 5 further simulated seasons, restricted to the evaluation window of day-of-year. It exists in every world, seasonal ones included.
- *Truth* is the exact I(D; V | C) of each world. It is closed-form for linear-Gaussian worlds, and computed by Monte Carlo or quadrature otherwise.

**Protocol.**
- Development worlds (§1), seeds 20 000+.
- Gates are declared in §3 after the development reading and before the validation run.
- The validation worlds (§4) use new shapes, new parameters and seeds 300 000+.
- No NEON observation is used anywhere in this notebook.

In [1]:
import os
os.environ.setdefault('OPENBLAS_NUM_THREADS', '1')  # small ridge problems: one BLAS thread per process is ~7x faster here

import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from scipy import special
from scipy.special import logsumexp
from scipy.stats import norm

from visual_hygrometry import data, inference, policy, scoring, splits, synthetic, worlds
from visual_hygrometry.synthetic import Node
from visual_hygrometry.worlds import cyc

pd.set_option('display.precision', 3)
par = lambda: Parallel(n_jobs=16, max_nbytes=None)
DOY0, N_DAYS, PER_DAY, HOURS, N_FUT = 152, 92, 14, (6, 20), 5
KNOTS, DOY_K, DOY_F, ILL_K = np.arange(-4, 45, 4.0), np.arange(140, 261, 15.0), np.arange(146, 251, 7.5), np.linspace(.1, 1.3, 7)  # fixed before any run
DESIGNS = {'1 season': 1, '3 seasons': 3}


def spec(gamma=0.0, slow=(0.0, 0.0), cloud=(0.0, 0.0, 0.0), r_persist=.5, v_persist=.5, r_day=.6, v_day=.4, beta=.4, illum=False):
    """Hourly SEM: diurnal T, moisture residual dres, D = mean(h) + beta T + dres + a S + c_D K, image cue
    V = gamma dres + b S + c_V K + noise. S: slow shared environmental state (day-level, AR 0.95 across days);
    K: cloudiness (lowers T, D and image brightness), observed as a clear-sky index only in illumination worlds."""
    (a, b), (cT, cD, cV) = slow, cloud
    s = {'slow': Node(sd=1.0, day=1.0, persist=.95), 'cloud': Node(sd=1.0, day=.6, ar=.8, persist=.5),
         'temp_cond_c': Node({'cloud': cT}, 3.5, mean=cyc(20, 5), day=.7, ar=.8, persist=.6),
         'dres': Node(sd=2.0, day=r_day, ar=.6, persist=r_persist),
         'd_ref_k': Node({'temp_cond_c': beta, 'dres': 1.0, 'slow': a, 'cloud': cD}, mean=cyc(7 - 20 * beta, 4 - 5 * beta)),
         'vis_moist': Node({'dres': gamma, 'slow': b, 'cloud': cV}, 1.0, day=v_day, ar=.5, persist=v_persist)}
    return s | ({'clearsky_k': Node({'cloud': -.15}, mean=lambda h: np.full_like(h, .7))} if illum else {})


def make(W, n_years, seed):
    """n_years independent seasons of N_DAYS days (day-of-year DOY0 ...), hour basis attached, world modification applied."""
    parts = []
    for y in range(n_years):
        df = synthetic.simulate(W['spec'], N_DAYS, PER_DAY, HOURS, seed=seed * 11 + y)
        parts.append(df.assign(year=y, doy=DOY0 + df.day, day=df.day + y * N_DAYS))
    df = pd.concat(parts, ignore_index=True)
    df = pd.concat([df, data.fourier(df.hour, 24, 2, 'hour')], axis=1).assign(hour_local=df.hour, ref_sensor_id='ref', cond_sensor_id='cond')
    return W['modify'](df, np.random.default_rng(seed + 7)) if 'modify' in W else df


def pipeline(df, W, ny, full=True):
    """The redesigned contrast on one simulated record; rows with year >= ny are future seasons (frozen target only)."""
    s, extra = (df.year < ny).to_numpy(), []
    if W.get('illum'):
        I = data.spline(df.clearsky_k, ILL_K, 'illum_spl')
        df, extra = pd.concat([df, I], axis=1), list(I)
    hs = W.get('hour_season', False)  # optional L4 level (post-hoc diagnostic, section 6)
    dfl, lv = worlds.add_context_levels(df, KNOTS, DOY_K, DOY_F, hs)
    top = lv[list(lv)[-1]] + extra
    _, _, y, _ = policy.select_sci(dfl, 'e2_illum' if W.get('illum') else 'e2', top, W['vis'])  # contract check, all levels
    y, V, Cnull = y.to_numpy(), df[W['vis']].to_numpy(), dfl[top].to_numpy()[s]
    levels = lambda d, sp: list(worlds.split_levels(d, sp, KNOTS, DOY_K, DOY_F, extra, hs)[0].values())
    g, yr, doy = df.day.to_numpy(), df.year.to_numpy(), df.doy.to_numpy()
    wk, raw = yr * 14 + (doy - DOY0) // 7, df[['hour_local', 'temp_cond_c']].to_numpy()
    if ny == 1:
        t = pd.Series(pd.to_datetime(g[s], unit='D'))
        sp = splits.kfold(splits.assign_folds(g[s] // 16, 5, 'contiguous'), t, t, embargo='3D')
    else:
        sp = splits.kfold(yr[s])
    keep = scoring.support_mask(raw[s], sp)
    D, chosen = inference.select_capacity(levels(df[s].reset_index(drop=True), sp), y[s], sp, wk[s], robust=True)
    r = scoring.contrast(D, [np.c_[d, v] for d, v in zip(D, inference.clip_to_training(V[s], sp))], y[s], sp, wk[s], keep=keep, robust=True)
    est_lo, est_hi = inference.cluster_interval(r.d, wk[s][r.scored])
    out = dict(est=r.mean, est_lo=est_lo, est_hi=est_hi, excluded=1 - keep.mean(), level=np.mean(chosen))
    if ny == 1:
        f, e = splits.chrono(g[s], .6, gap=2)
        fit, ev = np.isin(np.arange(len(g)), np.flatnonzero(s)[f]), np.isin(np.arange(len(g)), np.flatnonzero(s)[e])
        fut = ~s & (doy >= doy[ev].min())
    else:
        fit, ev, fut = s & (yr < ny - 1), s & (yr == ny - 1), ~s
    spc = [(fit, ev | fut)]
    Dc, ch = inference.select_capacity(levels(df, spc), y, spc, wk, robust=True)
    rc = scoring.contrast(Dc, [np.c_[Dc[0], inference.clip_to_training(V, spc)[0]]], y, spc, wk, keep=scoring.support_mask(raw, spc), robust=True)
    dz = np.full(len(y), np.nan)
    dz[rc.scored] = rc.d
    e = ev & np.isfinite(dz)
    lo, hi = inference.cluster_interval(dz[e], wk[e])
    day_sum, day_n = pd.Series(dz[e]).groupby(g[e]).agg(['sum', 'count']).to_numpy().T
    G = dz[e].mean()
    out |= dict(G=G, G_lo=lo, G_hi=hi, target=np.nanmean(dz[fut]), eval_weeks=len(np.unique(wk[e])), level_chrono=ch[0],
                dG_rare_frozen=dz[e & (y > np.quantile(y[e], .01))].mean() - G,
                day_infl=np.max(np.abs((dz[e].sum() - day_sum) / (e.sum() - day_n) - G)))
    if not full: return out
    _, p, null = inference.perm_test(D, V[s], y[s], sp, wk[s], df.hour_local.to_numpy()[s], keep=keep, units=g[s], strata=yr[s] if ny > 1 else None,
                                     Cnull=Cnull, shift_min=7, clip=.005, robust=True)
    raw3 = df[['hour_local', 'temp_cond_c', 'doy'] + (['clearsky_k'] if W.get('illum') else [])].to_numpy()
    flex = scoring.crossfit(raw3[s], y[s], sp, wk[s], model=inference.boost, robust=True)[2] - r.pred_c[2]
    ok = keep & np.isfinite(flex)
    return out | dict(p=p, n_shifts=len(null), null_median=np.median(null), flex_flag=inference.cluster_lower(flex[ok], wk[s][ok]) > 0, flex_mean=flex[ok].mean())


def one_rep(W, design, seed):
    ny = DESIGNS[design]
    df = make(W, ny + N_FUT, seed)
    out = dict(seed=seed) | pipeline(df, W, ny)
    rng, s = np.random.default_rng(seed + 3), (df.year < ny).to_numpy()
    drop = s & (rng.random(len(df)) < .01)
    rare = s & (df.d_ref_k.to_numpy() <= np.quantile(df.d_ref_k[s], .01))
    a, b = pipeline(df[~drop].reset_index(drop=True), W, ny, full=False), pipeline(df[~rare].reset_index(drop=True), W, ny, full=False)
    return out | dict(dest_rand=a['est'] - out['est'], dG_rand=a['G'] - out['G'], dG_rare_refit=b['G'] - out['G'])


def safe_rep(W, design, seed):
    """A replicate that raises is recorded with its error, counted in `failed`, and never silently dropped."""
    try: return one_rep(W, design, seed)
    except Exception as e: return dict(seed=seed, error=f'{type(e).__name__}: {e}')


def run(worlds_, seeds, stride=0):
    """stride > 0 gives every world its own seeds (seed + stride * world index), so worlds are independent evidence."""
    return {(k, d): pd.DataFrame(par()(delayed(safe_rep)(W, d, s + stride * i) for s in seeds)) for i, (k, W) in enumerate(worlds_.items()) for d in DESIGNS}


q90 = lambda x: np.quantile(np.abs(x), .9)


def summarise(worlds_, res):
    rows = []
    for (k, d), a in res.items():
        t, failed = worlds_[k]['truth'], int(a['error'].notna().sum()) if 'error' in a else 0
        a = a[a['error'].isna()] if 'error' in a else a
        rows.append(dict(world=k, design=d, truth=t, rej=(a.p <= .05).mean(), est=a.est.mean(), est_median=a.est.median(), bias=a.est.mean() - t,
                         cover_truth=((a.est_lo <= t) & (a.est_hi >= t)).mean(),
                         G=a.G.mean(), target=a.target.mean(), cover_target=((a.G_lo <= a.target) & (a.G_hi >= a.target)).mean(),
                         dG_rand_q90=q90(a.dG_rand), dest_rand_q90=q90(a.dest_rand), dG_rare_refit_q90=q90(a.dG_rare_refit), dG_rare_frozen_q90=q90(a.dG_rare_frozen),
                         day_infl=a.day_infl.median(), flex_flag=a.flex_flag.mean(), null_median=a.null_median.mean(), level=a.level.mean(), n_shifts=a.n_shifts.iloc[0], failed=failed))
    return pd.DataFrame(rows).set_index(['world', 'design'])

## 0 · Truths of the non-Gaussian positive worlds

- **Regime-specific information.** On humid-regime days (probability p), the moisture residual has a small SD and the image sees it. On dry days the residual is wide and the image sees nothing. I(D; V | C) is computed by Monte Carlo from the exact mixture densities.
- **Rare near-saturation information.** The image responds only when D falls below a threshold θ: V = a·σ((θ − D)/w) + noise. I(D; V | C) = E_c[h(V | c)] − h(noise), with p(v | c) obtained by Gauss–Hermite quadrature over D | C and a fine grid in v.

In [2]:
def truth_regime(p, sds, g, n=2_000_000, seed=0):
    rng = np.random.default_rng(seed)
    hum = rng.random(n) < p
    r = np.where(hum, sds[1], sds[0]) * rng.standard_normal(n)
    v = np.where(hum, g[1], g[0]) * r + rng.standard_normal(n)
    lw, sd, gg = np.log([1 - p, p]), np.array(sds), np.array(g)
    joint = logsumexp(lw + norm.logpdf(r[:, None], 0, sd) + norm.logpdf(v[:, None], gg * r[:, None], 1), axis=1)
    return np.mean(joint - logsumexp(lw + norm.logpdf(r[:, None], 0, sd), axis=1) - logsumexp(lw + norm.logpdf(v[:, None], 0, np.sqrt(gg ** 2 * sd ** 2 + 1)), axis=1))


def truth_rare(sp, amp, theta, width, n_rows=3000, seed=0):
    df = synthetic.simulate(sp, n_rows // PER_DAY, PER_DAY, HOURS, seed=seed)
    mu, sd = synthetic.conditional(sp, df, 'd_ref_k', ['temp_cond_c'])
    x, w = np.polynomial.hermite_e.hermegauss(80)
    v, h = np.linspace(-7, amp + 7, 1400), []
    for k in range(0, len(mu), 200):
        m = amp * special.expit((theta - (mu[k:k + 200, None] + sd[k:k + 200, None] * x)) / width)  # rows x nodes
        pv = (w / w.sum() * norm.pdf(v[:, None, None] - m[None])).sum(-1)  # grid x rows
        h.append(-np.trapezoid(pv * np.log(np.maximum(pv, 1e-300)), v, axis=0))
    return np.mean(np.concatenate(h)) - 0.5 * np.log(2 * np.pi * np.e)


print('check, linear-Gaussian limit of the regime truth (equal regimes):', round(truth_regime(.5, (2, 2), (.3, .3)), 4), 'vs exact', round(0.5 * np.log1p(.09 * 4), 4))

check, linear-Gaussian limit of the regime truth (equal regimes): 0.1539 vs exact 0.1537


## 1 · Development worlds

92 days of 14 hourly rows (06–19 h). Every world is run under both designs, 40 replicates each.

| world | structure | truth |
|---|---|---|
| N-base | no image–moisture link | 0 |
| N-persist | moisture residual and image noise persist across days (AR 0.95, CPER-like) | 0 |
| N-season | warming T, drying trend in D, image greening (tanh in day of year) | 0 |
| N-onset | D drops 2 K within ~6 days at "monsoon onset" (day 195); image greening | 0 |
| N-camera | camera service at mid-record: image gain × 1.5, offset + 1 | 0 |
| N-hetero | Var(D \| C) varies with T and hour | 0 |
| N-T | D += bump in T; image encodes the bump | 0 |
| N-hT | D += T curvature varying with hour; image encodes it | 0 |
| N-sT | D += season × T interaction; image encodes it | 0 |
| N-illum | cloudiness drives T, D and image brightness; clear-sky index in C (E2_illum) | 0 |
| P-lin | image sees the moisture residual | 0.05 |
| P-slow | slow shared state S (AR 0.95 across days) in D and in the image | exact |
| P-illum | the N-illum world under E2 (clear-sky index not in C) | exact |
| P-regime | information only on humid-regime days | Monte Carlo |
| P-rare | information only near saturation (D < 1.5 K, about 2 % of rows) | quadrature |

In [3]:
def season(df, rng):
    u = (df.doy - DOY0) / N_DAYS
    df['temp_cond_c'] = df.temp_cond_c + 3 * np.sin(np.pi * u)
    df['d_ref_k'] = df.d_ref_k - 1.5 * u
    df['vis_moist'] = df.vis_moist + 1.5 * np.tanh((df.doy - 185) / 10)
    return df

def onset(df, rng):
    df['d_ref_k'] = df.d_ref_k - 2 * special.expit((df.doy - 195) / 1.5)
    df['vis_moist'] = df.vis_moist + 1.5 * np.tanh((df.doy - 190) / 8)
    return df

def camera(df, rng):
    after = df.day >= N_DAYS // 2  # service half-way through the first season; every later day is after it
    df.loc[after, 'vis_moist'] = 1.5 * df.loc[after, 'vis_moist'] + 1.0
    return df

def hetero(df, rng):
    T, h = df.temp_cond_c.to_numpy(), df.hour.to_numpy()
    df['d_ref_k'] = df.d_ref_k + df.dres * ((0.6 + 0.1 * np.abs(T - 20)) * (1 + 0.4 * np.cos(2 * np.pi * (h - 14) / 24)) - 1)
    return df

def encodes(dterm, v_scale, v_sd):
    def f(df, rng):
        x = dterm(df)
        df['d_ref_k'] = df.d_ref_k + x
        df['vis_ctx'] = v_scale * x + rng.normal(0, v_sd, len(df))
        return df
    return f

bump = lambda d: 1.5 * np.exp(-(d.temp_cond_c - 17) ** 2 / 6)
curv = lambda d: ((d.temp_cond_c - 20) ** 2 / 9 - 1) * np.cos(2 * np.pi * (d.hour - 12) / 24)
sxt = lambda d: 0.15 * (d.temp_cond_c - 20) * np.tanh((d.doy - 185) / 12)

def regime_info(p=.4, sds=(2.5, 1.2), g=(0.0, .6)):
    def f(df, rng):
        hum = (rng.random(df.day.max() + 1) < p)[df.day]
        r = np.where(hum, sds[1], sds[0]) * df.dres / 2.0
        df['d_ref_k'] = df.d_ref_k - df.dres + r
        df['vis_moist'] = df.vis_moist + np.where(hum, g[1], g[0]) * r
        return df
    return f

def rare_info(amp=2.0, theta=1.5, width=.3):
    def f(df, rng):
        df['vis_moist'] = df.vis_moist + amp * special.expit((theta - df.d_ref_k) / width)
        return df
    return f

cmi = lambda sp, given=('temp_cond_c',): synthetic.cmi(sp, 'd_ref_k', 'vis_moist', list(given))
CLOUD = (-1.0, -.8, -.7)
DEV = {'N-base': dict(spec=spec()), 'N-persist': dict(spec=spec(r_persist=.95, v_persist=.95, r_day=.8, v_day=.8)),
       'N-season': dict(spec=spec(), modify=season), 'N-onset': dict(spec=spec(), modify=onset), 'N-camera': dict(spec=spec(), modify=camera),
       'N-hetero': dict(spec=spec(), modify=hetero), 'N-T': dict(spec=spec(), modify=encodes(bump, 2 / 1.5, .4), vis=['vis_ctx']),
       'N-hT': dict(spec=spec(), modify=encodes(curv, 1.0, .3), vis=['vis_ctx']), 'N-sT': dict(spec=spec(), modify=encodes(sxt, 2.0, .3), vis=['vis_ctx']),
       'N-illum': dict(spec=spec(cloud=CLOUD, illum=True), illum=True, truth=cmi(spec(cloud=CLOUD, illum=True), ('temp_cond_c', 'clearsky_k'))),
       'P-lin': dict(spec=spec(.162)), 'P-slow': dict(spec=spec(slow=(1.0, .5))),
       'P-illum': dict(spec=spec(cloud=CLOUD, illum=True)),
       'P-regime': dict(spec=spec(), modify=regime_info(), truth=truth_regime(.4, (2.5, 1.2), (0.0, .6))),
       'P-rare': dict(spec=spec(), modify=rare_info(), truth=truth_rare(spec(), 2.0, 1.5, .3))}
DEV = {k: {'vis': ['vis_moist']} | W | {'truth': W.get('truth', cmi(W['spec']) if k.startswith('P') else 0.0)} for k, W in DEV.items()}
print({k: round(W['truth'], 4) for k, W in DEV.items()})
dev = summarise(DEV, run(DEV, range(20000, 20040)))
dev

{'N-base': 0.0, 'N-persist': 0.0, 'N-season': 0.0, 'N-onset': 0.0, 'N-camera': 0.0, 'N-hetero': 0.0, 'N-T': 0.0, 'N-hT': 0.0, 'N-sT': 0.0, 'N-illum': np.float64(0.0), 'P-lin': np.float64(0.0499), 'P-slow': np.float64(0.0204), 'P-illum': np.float64(0.0205), 'P-regime': np.float64(0.0338), 'P-rare': np.float64(0.0183)}


truth    rej        est  est_median       bias  \
world     design                                                          
N-base    1 season   0.000e+00  0.125  3.736e-03  -8.752e-04  3.736e-03   
          3 seasons  0.000e+00  0.100 -4.110e-04  -1.579e-04 -4.110e-04   
N-persist 1 season   0.000e+00  0.025  8.977e-03   9.076e-03  8.977e-03   
          3 seasons  0.000e+00  0.050 -4.399e-02  -3.511e-02 -4.399e-02   
N-season  1 season   0.000e+00  0.125  5.694e-03  -2.364e-05  5.694e-03   
          3 seasons  0.000e+00  0.100 -2.452e-04  -6.982e-04 -2.452e-04   
N-onset   1 season   0.000e+00  0.125  1.005e-02  -4.000e-04  1.005e-02   
          3 seasons  0.000e+00  0.100 -7.790e-04  -5.890e-04 -7.790e-04   
N-camera  1 season   0.000e+00  0.125  5.047e-03  -7.937e-04  5.047e-03   
          3 seasons  0.000e+00  0.075 -9.722e-04  -6.664e-04 -9.722e-04   
N-hetero  1 season   0.000e+00  0.100  1.299e-04  -4.861e-04  1.299e-04   
          3 seasons  0.000e+00  0.100 -4.314e-04  -2.252e-04 -4.314e-04   
N-T       1 season   0.000e+00  0.025  4.596e-05  -2.584e-04  4.596e-05   
          3 seasons  0.000e+00  0.125  3.146e-04  -7.471e-05  3.146e-04   
N-hT      1 season   0.000e+00  0.050  1.569e-02   9.144e-03  1.569e-02   
          3 seasons  0.000e+00  0.100  5.938e-03   3.327e-03  5.938e-03   
N-sT      1 season   0.000e+00  0.025  2.655e-02   3.519e-02  2.655e-02   
          3 seasons  0.000e+00  0.075  3.563e-03   1.467e-03  3.563e-03   
N-illum   1 season   2.220e-16  0.100  4.816e-03  -7.536e-04  4.816e-03   
          3 seasons  2.220e-16  0.150 -1.179e-04  -5.780e-04 -1.179e-04   
P-lin     1 season   4.991e-02  0.600  2.849e-02   2.899e-02 -2.142e-02   
          3 seasons  4.991e-02  1.000  4.950e-02   4.847e-02 -4.077e-04   
P-slow    1 season   2.041e-02  0.125  6.965e-03   3.492e-03 -1.345e-02   
          3 seasons  2.041e-02  0.650  2.029e-02   1.766e-02 -1.211e-04   
P-illum   1 season   2.050e-02  0.500  1.097e-06   9.226e-03 -2.050e-02   
          3 seasons  2.050e-02  0.925  2.112e-02   1.916e-02  6.164e-04   
P-regime  1 season   3.378e-02  0.325  5.524e-03   1.794e-03 -2.826e-02   
          3 seasons  3.378e-02  0.875  1.235e-02   1.053e-02 -2.142e-02   
P-rare    1 season   1.827e-02  0.250  5.769e-03  -4.443e-04 -1.250e-02   
          3 seasons  1.827e-02  0.100  4.408e-04  -3.026e-04 -1.783e-02   

                     cover_truth          G     target  cover_target  \
world     design                                                       
N-base    1 season         0.850 -1.135e-03 -7.861e-04         0.950   
          3 seasons        0.850 -9.612e-04 -1.283e-03         0.875   
N-persist 1 season         0.800 -3.081e-03 -5.234e-04         0.625   
          3 seasons        0.475 -5.097e-02 -2.425e-02         0.700   
N-season  1 season         0.725 -8.123e-03 -4.219e-03         0.950   
          3 seasons        0.900 -1.332e-03 -1.038e-03         0.825   
N-onset   1 season         0.850 -2.584e-04 -1.194e-03         0.950   
          3 seasons        0.925  6.661e-05 -4.076e-04         0.950   
N-camera  1 season         0.800 -2.449e-03 -7.876e-04         0.875   
          3 seasons        0.875 -1.773e-03 -1.061e-03         0.900   
N-hetero  1 season         0.850 -1.394e-03 -5.551e-04         0.900   
          3 seasons        0.900 -7.374e-04 -9.643e-04         0.850   
N-T       1 season         0.850 -4.874e-04 -5.855e-04         0.925   
          3 seasons        0.850 -1.347e-04 -9.430e-05         0.875   
N-hT      1 season         0.625  9.173e-03  1.357e-02         0.825   
          3 seasons        0.575  6.222e-03  5.951e-03         0.925   
N-sT      1 season         0.625  1.706e-02  2.258e-02         0.975   
          3 seasons        0.750  5.446e-03  3.879e-03         0.975   
N-illum   1 season         0.875 -7.201e-04 -1.890e-03         0.925   
          3 seasons        0.925 -4.812e-04 -1.159e-03         0.800   
P-lin     1 season         0.700  1.417e

### Check added after the development run: Type I error of the shift test

In the development table several null worlds reject at 0.10–0.125. But all worlds share seeds 20 000–20 039, so their rejections come from the same random draws and are not independent evidence. Here the simplest null is run with 160 new seeds, under the declared test and three variants: no clipping, the fixed L1 context with an L1 null generator, and every non-identity shift (the exact cyclic group).

In [4]:
def p_variants(W, seed):
    df = make(W, 1, seed)
    dfl, lv = worlds.add_context_levels(df, KNOTS, DOY_K, DOY_F)
    y, V, g = df.d_ref_k.to_numpy(), df[W['vis']].to_numpy(), df.day.to_numpy()
    wk, t = (df.doy.to_numpy() - DOY0) // 7, pd.Series(pd.to_datetime(g, unit='D'))
    sp = splits.kfold(splits.assign_folds(g // 16, 5, 'contiguous'), t, t, embargo='3D')
    keep, LS = scoring.support_mask(df[['hour_local', 'temp_cond_c']].to_numpy(), sp), list(worlds.split_levels(df, sp, KNOTS, DOY_K, DOY_F)[0].values())
    D, _ = inference.select_capacity(LS, y, sp, wk, robust=True)
    p = lambda C, Cn, **kw: inference.perm_test(C, V, y, sp, wk, df.hour_local.to_numpy(), keep=keep, units=g, Cnull=dfl[lv[Cn]].to_numpy(), robust=True,
                                                **({'shift_min': 7, 'clip': .005} | kw))[1]
    return {'declared': p(D, 'L3'), 'no clipping': p(D, 'L3', clip=None), 'L1 context, L1 null generator': p(LS[0], 'L1'), 'all non-identity shifts': p(D, 'L3', shift_min=1)}

tI = pd.DataFrame(par()(delayed(p_variants)(DEV['N-base'], s) for s in range(50000, 50160)))
pd.DataFrame({'rejection at 0.05': (tI <= .05).mean(), 'mean p': tI.mean()})

,rejection at 0.05,mean p
declared,0.056,0.485
no clipping,0.062,0.483
"L1 context, L1 null generator",0.075,0.514
all non-identity shifts,0.056,0.486


## 2 · Reading the development suite

*Written after the development run and the Type I check, before the validation run.*

**Fixes made during development** (each found on a single development replicate, not tuned on the tables):
- *Seasonal extrapolation.* The edge folds of the 1-season design lie outside the training days. Unsupported day-of-year basis columns, amplified by the SD floor, put the context mean 8 K off. As a result, a 1 % row deletion moved one estimate by 0.31 nats. Fixed by clamping day of year to each split's training range (`worlds.split_levels`).
- *Numerics.* One exactly rank-deficient design made LAPACK's divide-and-conquer SVD fail; it now falls back to QR iteration (`scoring._svd`).
- *Compute.* Single-threaded BLAS is about 7× faster for these small ridge problems.

**Zero-information test.**
- With 40 shared seeds, the null worlds reject at 0.025–0.15. The check above with 160 independent seeds gives 0.056 for the declared test in the base world.
- Variants: no clipping 0.062, L1 context 0.075, all non-identity shifts 0.056.
- Neither clipping nor the restriction to shifts of at least 7 days explains an excess. The validation gives every world its own seeds so its rejection rates are independent.

**3-season design.**
- *Linear-Gaussian positives:* bias −0.0004 (P-lin), −0.0001 (P-slow), +0.0006 (P-illum); power 1.00, 0.65 and 0.93 at truths 0.05, 0.020 and 0.021.
- *Context-nonlinearity nulls:* small positive estimates remain, +0.006 (hour-varying T curvature) and +0.004 (season × T). This is the context-repair term KL_C. The test is not affected (rejection 0.10 and 0.075), because the null generator keeps the same linear projection of V on the L3 basis.
- *Persistence null (AR 0.95):* the estimate is negative (−0.044) and the interval covers its target in only 0.70 of replicates.
- *Stability:* the 90th percentile change after a random 1 % deletion is ≤ 0.017 nats for the estimate and ≤ 0.024 for G in every world except the persistence null (0.042 / 0.071).
- *Coverage:* the chronological interval covers its frozen target in 0.80–0.95 of replicates. It describes one evaluation season, and between-season variation of G is not in it.
- *Non-Gaussian positives:* the regime world captures 0.012 of a true 0.034. The rare near-saturation world captures nothing (0.0004 of 0.018), because a linear mean in V cannot express information that lives in 2 % of rows.

**1-season design, ruled out for effect sizes.**
- *Power:* 0.60 at a truth of 0.05, 0.13 at 0.020.
- *Bias:* the estimate is biased low (P-lin 0.028 for 0.050).
- *Stability:* the estimate is unstable, with 90th-percentile changes of 0.03–0.10 nats after a random 1 % deletion in every world except N-T (0.004).
- Only its zero-information test stays usable.

**Adequacy diagnostic.**
- The boosted context model flags ≤ 0.10 of replicates in 1-season worlds and none in 3-season worlds, except 0.225 in the 1-season persistence null.
- It does not detect the small context inadequacies of N-hT and N-sT. It remains a secondary check.

**Before validation.** No method change follows from this reading. The gates in §3 encode what the development suite suggests the 3-season design can meet, and include one gate it may fail (G5, coverage).

## 3 · Validation gates (declared before the validation run)

The gates apply to the **3-season design**, the design planned for NEON. The 1-season design is reported alongside without gates; its development results already rule it out for effect sizes (see the reading above). There are 150 replicates per world, each world with its own seeds, so the Monte Carlo SE of a 0.05 rate is 0.018.

| id | gate (3 seasons) | worlds |
|---|---|---|
| G1 | shift-test Type I ≤ 0.085 | every null |
| G2 | power ≥ 0.80 | linear-Gaussian positives with truth ≥ 0.03 |
| G3 | \|mean estimate − truth\| ≤ 0.01 | every null and linear-Gaussian positive |
| G4 | week-cluster interval of the cross-fit estimate covers the truth ≥ 0.85 | nulls without a context non-linearity, and linear-Gaussian positives |
| G5 | chronological interval covers its frozen target ≥ 0.85 | every world except the AR 0.9 persistence null |
| G6 | 90th percentile of \|Δ estimate\| and of \|Δ G\| ≤ 0.03 nats after deleting a random 1 % of rows and refitting everything | every world except the persistence null |
| G7 | 90th percentile of \|Δ G\| ≤ 0.05 nats after deleting the 1 % of rows with the lowest D and refitting | every world except the persistence null and the rare-information world |
| G8 | mean estimate ≤ truth + 0.005 (captured information never exceeds the truth) | non-Gaussian positives |
| G9 | context-adequacy diagnostic flags ≤ 0.10 | every world |

## 4 · Validation worlds (new shapes, parameters and seeds)

| world | structure | truth |
|---|---|---|
| V-N1 | base, β 0.5, day-level persistence 0.6 | 0 |
| V-N2 | moisture residual and image noise persistence 0.9 | 0 |
| V-N3 | T warming (1 − cos), D sinusoid over the season, linear-quadratic greening | 0 |
| V-N4 | D drops 2.5 K within ~4 days at day 210; image darkens over ~2 weeks | 0 |
| V-N5 | camera services at day 30 (gain 0.7, offset −1) and at the start of season 3 (gain 1.3, offset +0.5) | 0 |
| V-N6 | Var(D \| C) varying with hour and season | 0 |
| V-N7 | D += 0.4 \|T − 21\|^1.3; image encodes it | 0 |
| V-N8 | D += 1.2 tanh((T − 20)/2) sin(hour); image encodes it | 0 |
| V-N9 | D += season × hour interaction; image encodes it | 0 |
| V-N10 | cloudiness (other coefficients), clear-sky index in C (E2_illum) | 0 |
| V-P1, V-P2 | image sees the moisture residual | 0.03, 0.08 |
| V-P3 | slow shared state, stronger loadings | exact |
| V-P4 | V-N10 under E2 | exact |
| V-P5 | regime-specific information (p 0.3) | Monte Carlo |
| V-P6 | near-saturation information (θ 2 K, amplitude 1.5) | quadrature |

In [5]:
vspec = lambda **kw: spec(**({'beta': .5, 'r_persist': .6, 'v_persist': .6, 'r_day': .5, 'v_day': .5} | kw))

def season_v(df, rng):
    u = (df.doy - DOY0) / N_DAYS
    df['temp_cond_c'] = df.temp_cond_c + 2.5 * (1 - np.cos(np.pi * u))
    df['d_ref_k'] = df.d_ref_k + np.sin(2 * np.pi * u)
    df['vis_moist'] = df.vis_moist + 1.2 * u - 0.8 * u ** 2
    return df

def onset_v(df, rng):
    df['d_ref_k'] = df.d_ref_k - 2.5 * special.expit((df.doy - 210) / 1.0)
    df['vis_moist'] = df.vis_moist - 1.2 * special.expit((df.doy - 205) / 6)
    return df

def camera_v(df, rng):
    a, b = df.day >= 30, df.day >= 2 * N_DAYS
    df.loc[a, 'vis_moist'] = 0.7 * df.loc[a, 'vis_moist'] - 1.0
    df.loc[b, 'vis_moist'] = 1.3 * df.loc[b, 'vis_moist'] + 0.5
    return df

def hetero_v(df, rng):
    sc = (0.7 + 0.5 * np.sin(2 * np.pi * (df.hour - 10) / 24) ** 2) * (1 + 0.3 * np.tanh((df.doy - 200) / 20))
    df['d_ref_k'] = df.d_ref_k + df.dres * (sc - 1)
    return df

kink = lambda d: 0.4 * np.abs(d.temp_cond_c - 21) ** 1.3
tsin = lambda d: 1.2 * np.tanh((d.temp_cond_c - 20) / 2) * np.sin(2 * np.pi * (d.hour - 8) / 24)
sxh = lambda d: 0.8 * np.sin(2 * np.pi * (d.hour - 9) / 24) * np.tanh((d.doy - 200) / 15)
CLOUD_V = (-.6, -1.0, -.9)
gam = lambda c: np.sqrt(np.expm1(2 * c)) / 2  # moisture residual SD 2, image noise SD 1
VAL = {'V-N1': dict(spec=vspec()), 'V-N2': dict(spec=vspec(r_persist=.9, v_persist=.9, r_day=.8, v_day=.8)),
       'V-N3': dict(spec=vspec(), modify=season_v), 'V-N4': dict(spec=vspec(), modify=onset_v), 'V-N5': dict(spec=vspec(), modify=camera_v),
       'V-N6': dict(spec=vspec(), modify=hetero_v), 'V-N7': dict(spec=vspec(), modify=encodes(kink, .5, .4), vis=['vis_ctx']),
       'V-N8': dict(spec=vspec(), modify=encodes(tsin, 1.0, .3), vis=['vis_ctx']), 'V-N9': dict(spec=vspec(), modify=encodes(sxh, 1.5, .3), vis=['vis_ctx']),
       'V-N10': dict(spec=vspec(cloud=CLOUD_V, illum=True), illum=True, truth=cmi(vspec(cloud=CLOUD_V, illum=True), ('temp_cond_c', 'clearsky_k'))),
       'V-P1': dict(spec=vspec(gamma=gam(.03))), 'V-P2': dict(spec=vspec(gamma=gam(.08))), 'V-P3': dict(spec=vspec(slow=(1.2, .7))),
       'V-P4': dict(spec=vspec(cloud=CLOUD_V, illum=True)),
       'V-P5': dict(spec=vspec(), modify=regime_info(.3, (2.2, 1.0), (0.0, .8)), truth=truth_regime(.3, (2.2, 1.0), (0.0, .8), seed=1)),
       'V-P6': dict(spec=vspec(), modify=rare_info(1.5, 2.0, .4), truth=truth_rare(vspec(), 1.5, 2.0, .4, seed=1))}
VAL = {k: {'vis': ['vis_moist']} | W | {'truth': W.get('truth', cmi(W['spec']) if '-P' in k else 0.0)} for k, W in VAL.items()}
print({k: round(W['truth'], 4) for k, W in VAL.items()})
val = summarise(VAL, run(VAL, range(300000, 300150), stride=1000))
val

{'V-N1': 0.0, 'V-N2': 0.0, 'V-N3': 0.0, 'V-N4': 0.0, 'V-N5': 0.0, 'V-N6': 0.0, 'V-N7': 0.0, 'V-N8': 0.0, 'V-N9': 0.0, 'V-N10': np.float64(-0.0), 'V-P1': np.float64(0.03), 'V-P2': np.float64(0.08), 'V-P3': np.float64(0.0455), 'V-P4': np.float64(0.045), 'V-P5': np.float64(0.0253), 'V-P6': np.float64(0.0151)}


truth    rej        est  est_median       bias  \
world design                                                          
V-N1  1 season   0.000e+00  0.060 -4.458e-03  -9.694e-04 -4.458e-03   
      3 seasons  0.000e+00  0.060 -9.161e-04  -8.468e-04 -9.161e-04   
V-N2  1 season   0.000e+00  0.060  7.673e-03  -3.909e-04  7.673e-03   
      3 seasons  0.000e+00  0.100 -4.597e-03  -7.164e-03 -4.597e-03   
V-N3  1 season   0.000e+00  0.060  1.881e-04  -1.366e-03  1.881e-04   
      3 seasons  0.000e+00  0.047 -1.695e-03  -1.436e-03 -1.695e-03   
V-N4  1 season   0.000e+00  0.087  1.964e-03   5.739e-04  1.964e-03   
      3 seasons  0.000e+00  0.047 -8.831e-04  -5.975e-04 -8.831e-04   
V-N5  1 season   0.000e+00  0.033 -2.135e-03  -1.682e-03 -2.135e-03   
      3 seasons  0.000e+00  0.060 -1.128e-03  -8.251e-04 -1.128e-03   
V-N6  1 season   0.000e+00  0.060 -1.228e-04  -5.657e-04 -1.228e-04   
      3 seasons  0.000e+00  0.067 -1.010e-03  -9.277e-04 -1.010e-03   
V-N7  1 season   0.000e+00  0.040  1.024e-02   1.903e-03  1.024e-02   
      3 seasons  0.000e+00  0.080 -5.286e-04  -6.711e-04 -5.286e-04   
V-N8  1 season   0.000e+00  0.093  2.123e-03   6.222e-04  2.123e-03   
      3 seasons  0.000e+00  0.107  6.442e-04   2.960e-04  6.442e-04   
V-N9  1 season   0.000e+00  0.447  1.473e-02   1.058e-02  1.473e-02   
      3 seasons  0.000e+00  0.787  8.824e-03   8.497e-03  8.824e-03   
V-N10 1 season  -2.220e-16  0.047  1.342e-04  -1.549e-03  1.342e-04   
      3 seasons -2.220e-16  0.073 -3.256e-04  -8.688e-04 -3.256e-04   
V-P1  1 season   3.000e-02  0.560  2.186e-02   2.450e-02 -8.141e-03   
      3 seasons  3.000e-02  0.927  2.653e-02   2.369e-02 -3.466e-03   
V-P2  1 season   8.000e-02  0.707  5.030e-02   4.791e-02 -2.970e-02   
      3 seasons  8.000e-02  1.000  7.358e-02   7.325e-02 -6.423e-03   
V-P3  1 season   4.554e-02  0.120  1.516e-02   4.868e-03 -3.037e-02   
      3 seasons  4.554e-02  0.793  3.898e-02   3.098e-02 -6.558e-03   
V-P4  1 season   4.499e-02  0.547  1.685e-02   2.210e-02 -2.814e-02   
      3 seasons  4.499e-02  1.000  4.247e-02   4.172e-02 -2.515e-03   
V-P5  1 season   2.526e-02  0.207  1.134e-02   1.497e-03 -1.392e-02   
      3 seasons  2.526e-02  0.547  4.920e-03   4.446e-03 -2.034e-02   
V-P6  1 season   1.508e-02  0.127 -1.396e-03   5.681e-04 -1.648e-02   
      3 seasons  1.508e-02  0.220  1.201e-03   3.774e-04 -1.388e-02   

                 cover_truth          G     target  cover_target  dG_rand_q90  \
world design                                                                    
V-N1  1 season         0.887  2.261e-03  1.975e-03         0.920        0.032   
      3 seasons        0.887 -1.588e-03 -1.257e-03         0.913        0.004   
V-N2  1 season         0.787  1.778e-02  2.231e-02         0.740        0.149   
      3 seasons        0.707 -7.956e-03 -6.854e-03         0.787        0.029   
V-N3  1 season         0.847 -3.542e-03 -5.022e-03         0.867        0.031   
      3 seasons        0.860 -2.151e-03 -1.482e-03         0.907        0.003   
V-N4  1 season         0.867 -6.510e-03 -3.871e-03         0.927        0.072   
      3 seasons        0.900 -8.346e-04 -8.822e-04         0.900        0.004   
V-N5  1 season         0.840  1.935e-03  1.034e-03         0.880        0.019   
      3 seasons        0.873 -1.924e-03 -1.289e-03         0.927        0.005   
V-N6  1 season         0.907  3.724e-03  3.809e-03         0.833        0.031   
      3 seasons        0.887 -2.083e-04 -3.538e-04         0.900        0.006   
V-N7  1 season         0.880  6.199e-03  6.866e-03         0.880        0.025   
      3 seasons        0.913 -7.316e-04 -8.159e-04         0.940        0.006   
V-N8  1 season         0.893  2.954e-03  3.501e-03         0.907        0.008   
      3 seasons        0.967  8.978e-04  5.115e-04         0.893        0.004   
V-N9  1 season         0.693 -9.784e-03 -1.078e-02         0.927        0.028   
      3 seasons        0.200  9.941e-03  9.550e-03         0.933   

In [6]:
v = lambda w, c: val.loc[(w, '3 seasons'), c]
nulls, linpos, nonpos = [f'V-N{i}' for i in range(1, 11)], ['V-P1', 'V-P2', 'V-P3', 'V-P4'], ['V-P5', 'V-P6']
adequate, allw = ['V-N1', 'V-N2', 'V-N3', 'V-N4', 'V-N5', 'V-N6', 'V-N10'] + linpos, nulls + linpos + nonpos
gate = lambda vals, ok: (np.round(vals, 3), bool(ok))
not2, not26 = [w for w in allw if w != 'V-N2'], [w for w in allw if w not in ('V-N2', 'V-P6')]
pow_w = [w for w in linpos if VAL[w]['truth'] >= .03]
stab = lambda w: max(v(w, 'dest_rand_q90'), v(w, 'dG_rand_q90'))
G = {'G1': gate([v(w, 'rej') for w in nulls], all(v(w, 'rej') <= .085 for w in nulls)),
     'G2': gate([v(w, 'rej') for w in pow_w], all(v(w, 'rej') >= .8 for w in pow_w)),
     'G3': gate([v(w, 'bias') for w in nulls + linpos], all(abs(v(w, 'bias')) <= .01 for w in nulls + linpos)),
     'G4': gate([v(w, 'cover_truth') for w in adequate], all(v(w, 'cover_truth') >= .85 for w in adequate)),
     'G5': gate([v(w, 'cover_target') for w in not2], all(v(w, 'cover_target') >= .85 for w in not2)),
     'G6': gate([stab(w) for w in not2], all(stab(w) <= .03 for w in not2)),
     'G7': gate([v(w, 'dG_rare_refit_q90') for w in not26], all(v(w, 'dG_rare_refit_q90') <= .05 for w in not26)),
     'G8': gate([v(w, 'est') - VAL[w]['truth'] for w in nonpos], all(v(w, 'est') <= VAL[w]['truth'] + .005 for w in nonpos)),
     'G9': gate([v(w, 'flex_flag') for w in allw], all(v(w, 'flex_flag') <= .1 for w in allw))}
pd.DataFrame(G, index=['values', 'pass']).T

,values,pass
G1,"[0.06, 0.1, 0.047, 0.047, 0.06, 0.067, 0.08, 0...",False
G2,"[0.927, 1.0, 0.793, 1.0]",False
G3,"[-0.001, -0.005, -0.002, -0.001, -0.001, -0.00...",True
G4,"[0.887, 0.707, 0.86, 0.9, 0.873, 0.887, 0.873,...",False
G5,"[0.913, 0.907, 0.9, 0.927, 0.9, 0.94, 0.893, 0...",False
G6,"[0.004, 0.003, 0.005, 0.006, 0.006, 0.006, 0.0...",True
G7,"[0.004, 0.003, 0.009, 0.005, 0.003, 0.004, 0.0...",True
G8,"[-0.02, -0.014]",True
G9,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",True


## 5 · Reading the validation

*Written after the validation run, before the post-hoc section was added.*

**The redesigned protocol fails validation: gates G1, G2, G4 and G5 fail; G3 and G6–G8 pass; G9 passes but carries no information.** No replicate failed numerically.

| gate | result (3 seasons) | outcome |
|---|---|---|
| G1 Type I ≤ 0.085 | 0.047–0.080 in seven nulls; **V-N2 (persistence 0.9) 0.100, V-N8 (tanh T × sin hour) 0.107, V-N9 (season × hour) 0.787** | fail |
| G2 power ≥ 0.80 | 0.927 (V-P1, truth 0.030), 1.000 (V-P2, 0.080), **0.793 (V-P3, slow state, 0.046)**, 1.000 (V-P4, 0.045) | fail, narrowly |
| G3 \|bias\| ≤ 0.01 | largest: V-N9 +0.009, V-P3 −0.007, V-P2 −0.006 | pass |
| G4 cross-fit interval covers truth ≥ 0.85 | 0.86–0.90 in adequate nulls, but **V-N2 0.71, V-P1 0.79, V-P2 0.79, V-P3 0.49**; V-P4 0.88 | fail |
| G5 chronological interval covers target ≥ 0.85 | 0.89–0.94 in every world except **V-P3 0.74** | fail |
| G6 stability after a random 1 % deletion | 90th percentile ≤ 0.027 nats | pass |
| G7 deletion of the lowest-D 1 % | 90th percentile ≤ 0.024 nats | pass |
| G8 captured ≤ truth + 0.005 | regime 0.005 of 0.025; near-saturation 0.001 of 0.015 | pass |
| G9 adequacy flags ≤ 0.10 | 0 in every world, **including V-N9** | pass, uninformative |

**What failed, and why.**
1. **Context capacity, again.**
   - V-N9's D carries a diurnal cycle that changes over the season, and the image encodes it. None of L1–L3 contains an hour × season term, so the image repairs the context.
   - The shift test rejects in 79 % of replicates at zero information, because its linear null generator on the L3 basis cannot reproduce that structure either.
   - This structure is realistic at NEON. The hour basis is clock time, and sunrise and day length move by about an hour between June and August.
   - V-N8's 0.107 is a milder version of the same thing: the hour-varying T shape is representable by L3, but the capacity search often keeps a lower level (mean level 0.34).
2. **Interval coverage when information or residuals persist across weeks.**
   - In V-P3 the shared slow state has AR 0.95 across days. Its information arrives in multi-week blocks, so week clusters are not independent and both intervals are too narrow (0.49 and 0.74).
   - V-N2 (AR 0.9) shows the same for the zero-information test (0.100).
   - With three seasons there are only three independent units of that kind.
3. **Shrinkage.** In V-P1 and V-P2 the estimate is 12 % and 8 % below the truth, and the interval, which targets the information captured by the fitted pair, misses the truth in about one replicate in five.
4. **The adequacy diagnostic has no power.** The boosted context model never beats the ridge context by a confident margin, even where the test fails at 0.79. It cannot serve as a guard.

**What passed.**
- The point estimate is unbiased to within 0.01 nats in every null and linear-Gaussian positive. This includes persistence, seasons, a sharp onset, two camera services, a heteroscedastic D and the illumination null under E2_illum.
- It is stable under 1 % deletions.
- It never claims more than the true information in the non-Gaussian worlds.
- E2 and E2_illum separate exactly as designed: V-P4 captures 0.042 of 0.045, and V-N10 is null.

**1-season design (no gates, for comparison).**
- Type I 0.033–0.093, except V-N9 at 0.447.
- Power 0.56, 0.71, 0.12 and 0.55 in V-P1 to V-P4; positives biased by −0.008 to −0.030.
- 90th-percentile instability up to 0.15 nats.
- It is not usable for effect sizes; only its test is.

## 6 · Post-hoc diagnostic (added after the validation run; not a re-validation)

The worst gate failure is V-N9, whose D carries a season × hour interaction that the image encodes. None of the context levels L1–L3 can represent a diurnal cycle that changes over the season. Here an extra level L4 adds the hour basis × day-of-year spline (`hour_season=True`).
- V-N9 and V-N8 are re-run with and without L4; V-P1 is re-run with L4, to check the power cost.
- 3-season design, 60 new replicates each (seeds 400 000+).
- `flex_mean` is the mean held-out log-score gain of the boosted context model over the chosen ridge context. It shows whether the adequacy diagnostic sees anything at all.

These worlds have now been seen, so a level added here must be validated on new worlds before use.

In [7]:
POST = {'V-N9': VAL['V-N9'], 'V-N9 + L4': VAL['V-N9'] | {'hour_season': True}, 'V-N8': VAL['V-N8'], 'V-N8 + L4': VAL['V-N8'] | {'hour_season': True}, 'V-P1 + L4': VAL['V-P1'] | {'hour_season': True}}
post = {(k, '3 seasons'): pd.DataFrame(par()(delayed(safe_rep)(W, '3 seasons', s + 1000 * i) for s in range(400000, 400060))) for i, (k, W) in enumerate(POST.items())}
summarise(POST, post).join(pd.Series({k: a.flex_mean.mean() for (k, _), a in post.items()}, name='flex_mean'), on='world')[['truth', 'rej', 'est', 'bias', 'cover_truth', 'level', 'flex_flag', 'flex_mean', 'failed']]

,,truth,rej,est,bias,cover_truth,level,flex_flag,flex_mean,failed
world,design,,,,,,,,,
V-N9,3 seasons,0.00,0.767,7.664e-03,7.664e-03,0.250,0.672,0.0,-0.097,0
V-N9 + L4,3 seasons,0.00,0.050,5.459e-03,5.459e-03,0.433,1.361,0.0,-0.095,0
V-N8,3 seasons,0.00,0.083,5.094e-04,5.094e-04,0.967,0.422,0.0,-0.102,0
V-N8 + L4,3 seasons,0.00,0.067,4.774e-04,4.774e-04,1.000,0.567,0.0,-0.103,0
V-P1 + L4,3 seasons,0.03,0.883,2.523e-02,-4.772e-03,0.783,0.422,0.0,-0.101,0


**Reading (post-hoc, new seeds).**
- *The failure reproduces.* Without L4, V-N9 rejects in 0.767 of replicates.
- *L4 fixes the test but not the estimate.* With L4 the test is calibrated (0.050). The estimate still carries +0.005 nats of context repair, and its interval covers zero in only 0.43 of replicates. The capacity search does not always choose L4 (mean level index 1.36 of 3).
- *Other worlds.* V-N8 moves from 0.083 to 0.067. In V-P1, power is 0.883 with L4 on these seeds (0.927 without it in the validation, on other seeds), and the bias is −0.005.
- *The adequacy diagnostic cannot work as built.* The boosted context model scores about 0.10 nats *worse* than the ridge context in every world, so its lower bound never exceeds zero. A diagnostic must be at least as strong as the context model it audits.

An L4 level, or a solar-time hour basis, is a candidate change. It needs development and a fresh validation suite with solar-geometry-driven diurnal shifts before any use; these worlds have now been seen.